# QQQ Opening Range Bias — analysis notebook (v2)

Narrative walkthrough built on the tested `qqq_opening_bias` package. Every number
here is also reproducible non-interactively with:

```bash
python scripts/run_analysis.py --qqq data/QQQ_5min_10years_UTC.csv --nq data/nq-10y-1min.csv
```

⚠️ Requires the two CSVs in `../data/` (schema in `data/README.md`). Run top-to-bottom.

In [ ]:
import sys; sys.path.insert(0, "../src")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

from qqq_opening_bias import (
    BacktestConfig, run_backtest, run_placebo,
    load_qqq_bars, load_nq_bars,
    equity_series, compute_performance,
    per_trade_t_test, bootstrap_sharpe_ci, yearly_breakdown,
    slippage_sensitivity, equity_curves,
)

COMMISSION = 0.0005
SLIP = dict(entry_slippage_per_share=0.02, additional_stop_slippage_per_share=0.04)
BASE = dict(commission_per_share_per_side=COMMISSION)

## 1 · Load data
The replication runs on the full set of complete QQQ sessions — it is *not* pre-intersected with NQ, so the paper's ~1,795 trade count is recovered.

In [ ]:
qqq = load_qqq_bars("../data/QQQ_5min_10years_UTC.csv")
nq  = load_nq_bars("../data/nq-10y-1min.csv")
trading_dates = pd.to_datetime(qqq["date"]).dt.normalize().unique()
print(f"QQQ bars: {len(qqq):,}   NQ bars: {len(nq):,}   sessions: {len(trading_dates):,}")

## 2 · Scenarios

| scenario | costs | confirmation |
|---|---|---|
| replication | commission only | — |
| slippage | + $0.02 / $0.04 | — |
| NQ filter | + slippage | NQ 09:25 bar |
| placebo | + slippage | QQQ **own** 09:25 bar |

The placebo is the control: if the NQ filter were just two-bar momentum, the placebo would match it.

In [ ]:
results = {
    "replication": run_backtest(qqq, config=BacktestConfig(**BASE)),
    "slippage": run_backtest(qqq, config=BacktestConfig(**BASE, **SLIP)),
    "nq_filter": run_backtest(qqq, config=BacktestConfig(**BASE, require_nq_confirmation=True, **SLIP), nq_bars=nq),
    "placebo_qqq925": run_placebo(qqq, config=BacktestConfig(**BASE, **SLIP)),
}

rows = []
for name, r in results.items():
    eq = equity_series(r, trading_dates)
    perf = compute_performance(eq)
    edge = per_trade_t_test(r)
    lo, hi = bootstrap_sharpe_ci(eq)
    rows.append({"scenario": name, "trades": len(r.trades),
                 "net_pnl": round(r.cumulative_pnl),
                 "pnl_per_share": round(r.mean_pnl_per_share, 4),
                 "t_stat": round(edge.t_stat, 2), "sharpe": round(perf.sharpe, 2),
                 "sharpe_ci": f"[{lo:.2f}, {hi:.2f}]",
                 "cagr": round(perf.cagr, 3), "max_dd": round(perf.max_drawdown, 3)})
pd.DataFrame(rows).set_index("scenario")

## 3 · Equity curves vs buy & hold

In [ ]:
curves = equity_curves(qqq, nq)
fig, ax = plt.subplots(figsize=(11, 5))
for col in curves.columns:
    ax.plot(curves.index, curves[col], label=col, lw=1.6 if col in ("nq_filter","buy_hold") else 1.0)
ax.set_yscale("log"); ax.legend(); ax.grid(alpha=0.3)
ax.set_title("Equity curves (log scale)"); ax.set_ylabel("account value ($)")
plt.show()
curves.iloc[-1].round(0)

## 4 · Per-year breakdown
If the P&L concentrates in 2022, the edge is a volatility-regime artifact, not a structural one.

In [ ]:
for k in ("replication", "nq_filter"):
    print(f"— {k}")
    display(yearly_breakdown(results[k]).round(3))

## 5 · Slippage sensitivity
At what execution cost does the edge cross zero?

In [ ]:
sens = slippage_sensitivity(qqq, base_config=BacktestConfig(**BASE))
display(sens.round(4))
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(sens["entry_slippage"], sens["net_pnl"], marker="o")
ax.axhline(0, color="gray", lw=1)
ax.set_xlabel("entry slippage ($/share, stop = 2x)"); ax.set_ylabel("net PnL ($)")
ax.set_title("Edge vs execution cost"); ax.grid(alpha=0.3)
plt.show()

## 6 · Exit-reason anatomy
The +10R target is nearly decorative — most exits are stops or the session close.

In [ ]:
pd.DataFrame({name: pd.Series([t.exit_reason for t in r.trades]).value_counts(normalize=True)
              for name, r in results.items()}).round(3)